In [0]:
import boto3
import csv
import json
import io
import random
from datetime import datetime, timezone, date


### define env vars

In [0]:
# Accessing data directly through Spark/DBFS
BUCKET = 's3://fraud-demo-bucket-043309328060-us-east-1-an/'

### define exps 

In [0]:

COUNTRIES   = ["GB","US","EU","DE","FR","JP","CN","AU","CA"]
ENTITY_TYPES = ["INDIVIDUAL","ORGANISATION","VESSEL","AIRCRAFT"]
SANCTIONS_PROGRAMS = [
    "OFAC_SDN","UN_CONSOLIDATED","EU_CONSOLIDATED",
    "HMT_CONSOLIDATED","OFAC_CONSOLIDATED"
]


### define functions

In [0]:
import pandas as pd

# Generate MCC data as JSON
records = []
for i in range(200):
    records.append({
        "entity_id":        f"ENT{str(i+1).zfill(8)}",
        "entity_type":      random.choice(ENTITY_TYPES),
        "full_name":        f"Sanctioned Entity {i+1}",
        "aliases":          json.dumps([f"Alias {i+1}A", f"Alias {i+1}B"]),
        "country_of_origin":random.choice(COUNTRIES),
        "sanctions_program":random.choice(SANCTIONS_PROGRAMS),
        "listed_date":      "2023-06-15",
        "last_updated":     date.today().isoformat(),
        "is_active":        random.choice([True,True,True,False]),
        "risk_score":       round(random.uniform(0.5, 1.0), 3),
        "source_list":      random.choice(SANCTIONS_PROGRAMS),
    })

df = spark.createDataFrame(pd.DataFrame(records))
file_path = f"{BUCKET}reference/sanctions/{datetime.now(timezone.utc).strftime('%Y-%m-%dT%H%M%S')}.csv"
df.write.mode("append").option("header", True).csv(file_path)
print(f"Written {len(records)} sanction records to: {file_path}")